# Week 37, Day 6: Forge: Week 37 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w037/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
print("1.", round(1 - np.sqrt(1 - 2 * 0.3), 3))
print("2.", 1 + (10 - 1) * 0.2)
print("3.", 1 / 2 ** 6)

### Exercise: Part A: screening units across sessions

Using the helpers spikes_of, contamination and presence_ratio (given), write clean_units(f, T) returning a list of the indices of units that pass both criteria.

In [ ]:
import io
import itertools
import numpy as np
import h5py
import matplotlib.pyplot as plt

def make_session(seed):
    g = np.random.default_rng(seed); ntr = 300
    starts = np.cumsum(g.uniform(2.5, 3.5, ntr)); on = starts + 0.5
    con = g.choice([-1, -0.25, -0.0625, 0, 0, 0, 0.0625, 0.25, 1], ntr)
    choice = np.where(g.random(ntr) < 0.06 + 0.88 / (1 + np.exp(-8 * con)), 1, -1)
    areas = np.repeat(["VISp", "MOs", "CA1"], 10); T = starts[-1] + 3
    spikes, contam, lost = [], [], []
    for u in range(30):
        base = g.uniform(2, 8); a = areas[u]; w = g.uniform(0.5, 1.5)
        own = np.cumsum(g.exponential(1 / base, int(base * T * 1.3)) + 0.002); own = own[own < T]
        extra = []
        for o, c, ch in zip(on, con, choice):
            if a == "VISp":
                extra += list(o + g.uniform(0.05, 0.25, g.poisson(w * 3 * max(c, 0) + 0.2)))
            if a == "MOs":
                extra += list(o + g.uniform(0.3, 0.7, g.poisson(w * (2.0 if ch == 1 else 0.4))))
        st = np.sort(np.r_[own, np.array(extra)])
        cf = g.uniform(0.5, 1.0) if g.random() < 0.2 else 0.0
        if cf > 0:
            st = np.sort(np.r_[st, g.uniform(0, T, int(cf * len(st)))])
        lf = g.uniform(0.3, 0.6) if g.random() < 0.15 else 0.0
        if lf > 0:
            st = st[st < T * (1 - lf)]
        spikes.append(st); contam.append(cf); lost.append(lf)
    buf = io.BytesIO()
    with h5py.File(buf, "w") as w:
        w.attrs["subject_id"] = f"mouse-{seed}"
        tr = w.create_group("intervals/trials")
        tr["stim_on_time"] = on; tr["contrast"] = con; tr["choice"] = choice; tr["start_time"] = starts
        w.create_dataset("units/spike_times", data=np.concatenate(spikes))
        w.create_dataset("units/spike_times_index", data=np.cumsum([len(s) for s in spikes]))
        w.create_dataset("units/location", data=np.array(areas, dtype="S4"))
    return h5py.File(buf, "r"), T, (np.array(contam), np.array(lost))

sessions = [make_session(s) for s in range(101, 107)]
truth = [s[2] for s in sessions]

def spikes_of(f, k):
    idx = f["units/spike_times_index"]
    return f["units/spike_times"][0 if k == 0 else int(idx[k - 1]): int(idx[k])]

def contamination(st, T, tr=0.0015):
    m = np.sum(np.diff(st) < tr) / (2 * tr * len(st) ** 2 / T)
    return 1 - np.sqrt(1 - 2 * min(m, 0.4999))

def presence_ratio(st, T, block=60.0):
    nb = int(np.ceil(T / block))
    return np.mean(np.histogram(st, np.arange(nb + 1) * block)[0] > 0)

def clean_units(f, T):
    ...

kept = [clean_units(f, T) for f, T, _ in sessions]
for i, ks in enumerate(kept):
    loc = [s.decode() for s in sessions[i][0]["units/location"][:]]
    print(f"mouse {i + 1}: kept {len(ks)} of 30 units ({', '.join(f'{a} {sum(loc[k] == a for k in ks)}' for a in ('VISp', 'MOs', 'CA1'))})")

In [ ]:
import numpy as np
def test_counts():
    "between 14 and 26 units survive in every session"
    assert all(14 <= len(k) <= 26 for k in kept)

def test_bad_removed():
    "no unit with contamination of 50% or more survives"
    assert all(truth[i][0][k] < 0.5 for i, ks in enumerate(kept) for k in ks)

test_counts()
test_bad_removed()
print("All checks passed")

### Exercise: Part B: choice signals across mice

Complete count_matrix (spike counts of the given units in the window after each onset, trials by units) and signflip_p (the one-sided exact p-value: the fraction of the 2^n sign patterns applied to accuracy minus 0.5 whose mean is at least the observed mean).

In [ ]:
def logreg_cv(X, y, k=5, lam=1.0, seed=0):
    g = np.random.default_rng(seed); idx = g.permutation(len(y)); correct = 0
    for fold in np.array_split(idx, k):
        trn = np.setdiff1d(idx, fold); mu, sd = X[trn].mean(0), X[trn].std(0) + 1e-9
        A = np.c_[(X[trn] - mu) / sd, np.ones(len(trn))]; w = np.zeros(A.shape[1]); yy = (y[trn] > 0).astype(float)
        for _ in range(25):
            p = 1 / (1 + np.exp(-A @ w))
            w += np.linalg.solve((A * (p * (1 - p))[:, None]).T @ A + lam * np.eye(len(w)), A.T @ (yy - p) - lam * w)
        B = np.c_[(X[fold] - mu) / sd, np.ones(len(fold))]
        correct += np.sum(((B @ w) > 0) == (y[fold] > 0))
    return correct / len(y)

def count_matrix(spike_lists, onsets, t0=0.3, t1=0.7):
    ...

def signflip_p(accs):
    d = np.asarray(accs) - 0.5
    ...

acc = {"VISp": [], "MOs": [], "CA1": []}
for (f, T, _), ks in zip(sessions, kept):
    loc = [s.decode() for s in f["units/location"][:]]
    on = f["intervals/trials/stim_on_time"][:]; con = f["intervals/trials/contrast"][:]; ch = f["intervals/trials/choice"][:]
    zero = con == 0
    for a in acc:
        units = [spikes_of(f, k) for k in ks if loc[k] == a]
        acc[a].append(logreg_cv(count_matrix(units, on[zero]), ch[zero]))
pvals = {a: signflip_p(v) for a, v in acc.items()}
for a in acc:
    print(f"{a:5s} accuracy by mouse {np.round(acc[a], 2)}, mean {np.mean(acc[a]):.2f}, sign-flip p = {pvals[a]:.3f}")
plt.boxplot([acc[a] for a in acc]); plt.xticks([1, 2, 3], list(acc)); plt.axhline(0.5, ls="--", color="gray"); plt.ylabel("choice decoding (zero contrast)"); plt.show()

In [ ]:
import numpy as np
def test_mos():
    "MOs decodes choice above 70% on average and is significant across mice (p below 0.05)"
    assert np.mean(acc["MOs"]) > 0.7 and pvals["MOs"] < 0.05

def test_others():
    "VISp and CA1 stay below 60% on average and are not significant"
    assert np.mean(acc["VISp"]) < 0.6 and np.mean(acc["CA1"]) < 0.6
    assert pvals["VISp"] > 0.05 and pvals["CA1"] > 0.05

test_mos()
test_others()
print("All checks passed")